In [1]:
import json
import sys
from rich import print as rprint
from pathlib import Path
import re
from datetime import datetime

nb_dir = Path.cwd()
project_root = nb_dir.parent.parent
sys.path.insert(0, str(project_root))
from scripts.text_matching import normalise_text, test_similarity

In [2]:
matched_dict_file = Path(project_root / "scripts/notebooks/matched_from_nope.json")
with open(matched_dict_file, "r") as f:
   matched_dict = json.load(f)

parsed_data_file = Path(project_root / "scripts/notebooks/parsed_data_file.json")
with open(parsed_data_file, "r") as f:
   parsed_data = json.load(f)

db_books_file = Path(project_root / "data_reload/db_files/books.json")
with open(db_books_file, "r") as f:
   books = json.load(f)
books_dict = {i["composite_id"]: i for i in books}

In [ ]:


FUZZY_THRESHOLD = 90
ROLE_ORDER = ["is_author", "is_editor", "is_contributor", "is_translator"]

b2p_found_data = {}
b2p_found_list = []
b2p_not_found = {}

for composite_id, people in matched_dict.items():
    book_id = books_dict[composite_id]["book_id"]
    parsed = parsed_data[composite_id]

    role_entries = []
    for author in parsed["authors"] or []:
        role_entries.append(("is_author", author["display_name"]))
    for editor in parsed["editors"] or []:
        role_entries.append(("is_editor", editor["display_name"]))
    for contributor in parsed["contributors"] or []:
        role_entries.append(("is_contributor", contributor["display_name"]))
    if parsed["translator"]:
        role_entries.append(("is_translator", parsed["translator"]["display_name"]))

    rows = []
    role_counters = {role: 0 for role in ROLE_ORDER}

    for person in people:
        target = person["display_norm"]

        matches = [(flag, name) for flag, name in role_entries
                   if normalise_text(name) == target]

        if not matches:
            matches = [(flag, name) for flag, name in role_entries
                       if test_similarity(name, target) >= FUZZY_THRESHOLD]

        if not matches:
            b2p_not_found.setdefault(composite_id, []).append(person)
            continue

        flags = {flag for flag, name in matches}

        sort_order = None
        for role in ROLE_ORDER:
            if role in flags:
                role_counters[role] += 1
                if sort_order is None:
                    sort_order = role_counters[role]

        row = {
            "book_id": book_id,
            "composite_id": composite_id,
            "person_id": person["person_id"],
            "unified_id": person["unified_id"],
            "display_name": matches[0][1],
            "family_name": None,
            "given_names": None,
            "name_prefix": None,
            "name_particles": None,
            "name_suffix": None,
            "single_name": None,
            "sort_order": sort_order,
            "is_author": "is_author" in flags,
            "is_editor": "is_editor" in flags,
            "is_contributor": "is_contributor" in flags,
            "is_translator": "is_translator" in flags,
        }
        rows.append(row)
        b2p_found_list.append(row)

    if rows:
        b2p_found_data[composite_id] = rows


rprint(f"""
=== LOG ===
len data: {len(b2p_found_data)}
list: {len(b2p_found_list)}
not found: {len(b2p_not_found)}

=== DONE ===
""")


=== LOG ===
len data: 569
list: 734
not found: 61

=== DONE ===

In [4]:
with open("no_exact_match_found.json", "w") as f:
    json.dump(b2p_not_found, f, ensure_ascii=False, indent=2)


In [5]:
with open("b2p_found_data_02.json", "w") as f:
    json.dump(b2p_found_data, f, ensure_ascii=False, indent=2)

with open("b2p_found_list_02.json", "w") as f:
    json.dump(b2p_found_list, f, ensure_ascii=False, indent=2)